In [1]:
import json
import os

import numpy as np
import prettytable as pt
import pandas as pd
import pyarrow.parquet as pq
from scipy.optimize import curve_fit
from scipy.integrate import quad

import hist
# import matplotlib.pyplot as plt
# from sklearn.metrics import roc_curve, roc_auc_score
# from scipy.integrate import trapezoid

import eos_utils as eos

# Workspace packages
from HHtobbyy.event_discrimination.DFDataset import DFDataset
from HHtobbyy.workspace_utils import match_sample
from HHtobbyy.event_discrimination.models import map_model_to_Model


dfdataset_config = "/eos/uscms/store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manos_2026-04-20_12-49-04/dataset_config.json"
dfdataset = DFDataset(dfdataset_config)
model, model_config = "MLP", {"output_dirpath": "/uscms/home/tsievert/nobackup/XHYbbgg/Model_Outputs/ManosMLPv6/2026-04-20_13-59-41", "activation_func": "ELU"}
model = map_model_to_Model(model)(dfdataset, model_config)

/store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manos_2026-04-20_12-49-04/dataset_config.json


In [2]:
cats = {
    # "boosted": {
    #     "is_boosted": 1
    # },
    "vbfhh_cat1": {
        "AUX_DVBFHH": 0.774,  # >
    },
    "cat1": {
        "AUX_DggFHH": 0.99292177413867067,  # >
        "AUX_DnonRes": 0.42667485697897478,  # <
        "AUX_DRes": 0.90612735576941672,  # <
    },
    "cat2": {
        "AUX_DggFHH": 0.68115907008322718,  # >
        "AUX_DnonRes": 0.00064209200655934,  # <
        "AUX_DRes": 0.96341029269140221,  # <
    },
    "cat3" : {
        "AUX_DggFHH": 0.87036774207656309,  # >
        "AUX_DnonRes": 0.00221775294750575,  # <
        "AUX_DRes": 0.84544492824676509,  # <
    }
}

In [ ]:
filepaths = dfdataset.get_test_filepaths(0, regex='')
test_pre = None
for filepath in filepaths['test']:
    eos_filepath = eos.load_file_eos(filepath)
    table = pq.read_table(eos_filepath, filters=[[(col, '>' if col.endswith('HH') else '<', cut) for col, cut in cat.items()] for name, cat in cats.items()])
    if test_pre is None: test_pre = table.to_pandas()
    else: test_pre = pd.concat([test_pre, table.to_pandas()])
    eos.delete_lockfile(eos_filepath)

dipho_mass_window = np.logical_and(test_pre['AUX_mass'].gt(100).to_numpy(), test_pre['AUX_mass'].lt(180).to_numpy())
pho_mva_cut = test_pre['AUX_pass_mva-0.7'].gt(0).to_numpy()
snt_cuts = np.logical_and(dipho_mass_window, pho_mva_cut)
test = test_pre.loc[snt_cuts]



# expected_eras = {
#     # sim
#     'Run2_2016postVFP/sim', 'Run2_2016preVFP/sim', 'Run2_2017/sim', 'Run2_2018/sim',
#     'Run3_2022/sim/postEE', 'Run3_2022/sim/preEE', 'Run3_2023/sim/postBPix', 'Run3_2023/sim/preBPix', 'Run3_2024/sim', 'Run3_2025/sim',
#     # data
#     'Run2_2016postVFP/data', 'Run2_2016preVFP/data', 'Run2_2017/data', 'Run2_2018/data',
#     'Run3_2022/data', 'Run3_2023/data', 'Run3_2024/data', 'Run3_2025/data'
# }
# set_of_unique_sample_eras = set(pd.unique(test_pre['AUX_sample_era']).tolist())
# assert expected_eras == set_of_unique_sample_eras, f"sample eras do not match expected (either missing eras or names changed): \n {expected_eras} \n vs. \n {set_of_unique_sample_eras}"
# for sample_era in pd.unique(test_pre['AUX_sample_era']):
#     set_of_unique_sample_names = set(pd.unique(test_pre.loc[test_pre['AUX_sample_era'].eq(sample_era), 'AUX_sample_name']).tolist())
#     if 'data' in sample_era:
#         expected_set = {'Data'}
#     elif 'sim' in sample_era:
#         expected_set = {
#             'DDQCDGJets', 'GGJets', 'TTGG', 
#             'GluGluHToGG', 'VBFHToGG', 'VHToGG', 'ttHToGG', 'bbHToGG',
#             'GluGluToHH_kl1p00_kt1p00_c20p00', 'VBFToHH_cv1p00_c2v1p00_c31p00'
#         }
#         if '201' in sample_era: expected_set.remove('bbHToGG')
#         elif '2022' in sample_era: expected_set.remove('VBFToHH_cv1p00_c2v1p00_c31p00')
#     else:
#         raise Exception(f"Improper naming for era, expected to either contain \'sim\' or \'data\': {sample_era}")
#     assert set_of_unique_sample_names == expected_set, f"{sample_era} - samples names do not match expected: \n {expected_set} \n vs. \n {set_of_unique_sample_names}"
#     # for sample_name in pd.unique(test_pre.loc[test_pre['AUX_sample_era'].eq(sample_era), 'AUX_sample_name']):
#     #     print('-'*60, '\n', sample_name)


vbf23_mask = np.logical_and(
    test['AUX_sample_era'].isin(['Run3_2023/sim/preBPix', 'Run3_2023/sim/postBPix']),
    test['AUX_sample_name'].eq('VBFHH')
)  # makeup for lack of 2022 VBFHH signal 
test.loc[vbf23_mask, 'AUX_eventWeight'] = 1.78 * test.loc[vbf23_mask, 'AUX_eventWeight']


ArrowInvalid: No match for FieldRef.Name(AUX_DVBFHH) in eta: double
lead_eta: double
lead_phi: double
lead_mvaID: double
sublead_eta: double
sublead_phi: double
sublead_mvaID: double
nonResReg_vbfpair_lead_bjet_eta: double
nonResReg_vbfpair_lead_bjet_phi: double
nonResReg_vbfpair_lead_bjet_bTagWPL: double
nonResReg_vbfpair_lead_bjet_bTagWPM: double
nonResReg_vbfpair_lead_bjet_bTagWPT: double
nonResReg_vbfpair_lead_bjet_bTagWPXT: double
nonResReg_vbfpair_lead_bjet_bTagWPXXT: double
nonResReg_vbfpair_sublead_bjet_eta: double
nonResReg_vbfpair_sublead_bjet_phi: double
nonResReg_vbfpair_sublead_bjet_bTagWPL: double
nonResReg_vbfpair_sublead_bjet_bTagWPM: double
nonResReg_vbfpair_sublead_bjet_bTagWPT: double
nonResReg_vbfpair_sublead_bjet_bTagWPXT: double
nonResReg_vbfpair_sublead_bjet_bTagWPXXT: double
nonResReg_vbfpair_DeltaR_j1g1: double
nonResReg_vbfpair_DeltaR_j2g1: double
nonResReg_vbfpair_DeltaR_j1g2: double
nonResReg_vbfpair_DeltaR_j2g2: double
nonResReg_vbfpair_DeltaR_jg_min: doubl

Exception in thread Thread-232 (watch_tmp):
Traceback (most recent call last):
  File "/uscms/home/tsievert/nobackup/miniforge3/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/uscms/home/tsievert/nobackup/miniforge3/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/uscms_data/d3/tsievert/XHYbbgg/eos-utils/src/eos_utils/watch_tmp.py", line 25, in watch_tmp
    raise TimeoutError(f"ERROR: Tmp file watching for {tmp_filepath} timed out.")
TimeoutError: ERROR: Tmp file watching for .tmp_load-3107966632807878772.parquet timed out.


In [ ]:
manos_sample = pd.read_parquet('/uscms/home/tsievert/nobackup/XHYbbgg/v6_merged_scored_events.parquet', filters=[("year", "==", 2024), ("sample", "==", "GluGluToHH_kl-1p00_kt-1p00_c2-0p00")])

In [ ]:
#############################################################
# ASCii histogram for rapid plotting
def ascii_hist(x, bins=10, weights=None):
    N,X = np.histogram(x, bins=bins, weights=weights)
    width = 50
    nmax = np.max(N.max())
    for (xi, n) in zip(X,N):
        bar = '#'*int(n*width/nmax)
        xi = '{0: <8.4g}'.format(xi).ljust(10)
        print('{0}| {1}'.format(xi,bar))
def ascii_hist(x, bins=10, weights=None, fit=None):
    N,X = np.histogram(x, bins=bins, weights=weights)
    width = 50
    nmax = np.max([N.max(), fit.max()])
    if fit is None:
        for (xi, n) in zip(X,N):
            bar = '#'*int(n*width/nmax)
            xi = '{0: <8.4g}'.format(xi).ljust(10)
            print('{0}| {1}'.format(xi,bar))
    else:
        for (xi, n, fiti) in zip(X,N,fit):
            bar = '#'*int(n*width/nmax)
            if fiti > n: bar = bar + ' '*(int(fiti*width/nmax) - int(n*width/nmax)) + '+'
            else: bar = ''.join([bar[j] if j != int(fiti*width/nmax) else '+' for j in range(len(bar))])
            xi = '{0: <8.4g}'.format(xi).ljust(10)
            print('{0}| {1}'.format(xi,bar))

#############################################################
# Sideband fit functions for nonRes bkg estimaton
def exp_func(x, a, b):
    return a * np.exp(b * x)
def sd_hist(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float]):
    return hist.Hist(
        hist.axis.Regular(int((fit_bins[1]-fit_bins[0])//fit_bins[2]), fit_bins[0], fit_bins[1], name="var", growth=False, underflow=False, overflow=False), 
        storage='weight'
    ).fill(var=mass, weight=weight)
def exp_mass_fit(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float], sigma: bool=False):
    _hist_ = sd_hist(mass, weight, fit_bins)
    params, _ = curve_fit(
        exp_func, _hist_.axes.centers[0]-_hist_.axes.centers[0][0], _hist_.values(), p0=(_hist_.values()[0], -0.1), 
        sigma=np.where(_hist_.values() != 0, np.sqrt(_hist_.variances()), 0.76) if sigma else None
    )
    print(_hist_)
    ascii_hist(mass, bins=np.arange(fit_bins[0], fit_bins[1], fit_bins[2]), weights=weight, fit=exp_func(_hist_.axes.centers[0]-_hist_.axes.centers[0][0], a=params[0], b=params[1]))
    return _hist_, params
def est_yield(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float], sr_masscut: list[float], sigma: bool=False):
    _hist_, fit_params = exp_mass_fit(mass, weight, fit_bins, sigma=sigma)
    return quad(exp_func, sr_masscut[0]-_hist_.axes.centers[0][0], sr_masscut[1]-_hist_.axes.centers[0][0], args=tuple(fit_params))[0] / fit_bins[2]

In [ ]:
data_samples = [sample_name for sample_name in sorted(pd.unique(test['AUX_sample_name']).tolist()) if match_sample(sample_name, ['Data']) is not None]
sideband_samples = [sample_name for sample_name in sorted(pd.unique(test['AUX_sample_name']).tolist()) if match_sample(sample_name, ['GJet', 'TTG']) is not None]
non_sideband_samples = [sample_name for sample_name in sorted(pd.unique(test['AUX_sample_name']).tolist()) if sample_name not in data_samples+sideband_samples]
print(f"All samples: {sorted(pd.unique(test['AUX_sample_name']).tolist())}")
print(f"Data samples: {data_samples}")
print(f"nonRes MC samples: {sideband_samples}")
print(f"Res/Signal MC samples: {non_sideband_samples}")


# field_names = ['Category'] + non_sideband_samples + ['nonRes SB fit', 'data SB fit', 's/b with nonRes fit', 's/b with data fit']
# field_names = ['Category'] + non_sideband_samples + ['nonRes SB fit', 'data SB fit']
# field_names = ['Category'] + non_sideband_samples + sideband_samples + sideband_samples + ['nonRes SB fit', 'N Data SB', 'data SB fit']

field_names = ['Category'] + non_sideband_samples + sideband_samples + ['single-H', 'non-Res', 'N Data SB']
table = pt.PrettyTable(field_names=field_names, float_format=".5")

not_prev_cut_mask = {}
for i, (name, cuts) in enumerate(cats.items()):

    for eras in [['Run2_2016preVFP/sim', 'Run2_2016postVFP/sim', 'Run2_2016preVFP/data', 'Run2_2016postVFP/data'], ['Run2_2017/sim', 'Run2_2017/data'], ['Run2_2018/sim', 'Run2_2018/data'], ['Run3_2022/sim/preEE', 'Run3_2022/sim/postEE', 'Run3_2022/data'], ['Run3_2023/sim/preBPix', 'Run3_2023/sim/postBPix', 'Run3_2023/data'], ['Run3_2024/sim', 'Run3_2024/data'], ['Run3_2025/sim', 'Run3_2025/data'], ['tot']]:
        if 'tot' not in eras:
            era_mask = test['AUX_sample_era'].isin(eras); evtwt_factor = 1
            era_name = eras[-1].split('_')[-1].replace('/data', '').replace('postVFP', '')
        else:
            era_mask = np.ones(len(test), dtype=bool); evtwt_factor = 1
            era_name = 'tot'
        new_row = [name+' '+era_name]
        nonRes_sideband = pd.DataFrame({'mass': pd.Series(dtype='float'), 'weight_tot': pd.Series(dtype='float')})

        singleH_yield = 0.
        nonRes_yield = 0.

        for sample in non_sideband_samples+sideband_samples:

            sample_mask = np.logical_and(era_mask, test['AUX_sample_name'].eq(sample))
            if sample+era_name not in not_prev_cut_mask: not_prev_cut_mask[sample+era_name] = sample_mask

            pass_cut_mask = not_prev_cut_mask[sample+era_name]
            if i != 0:
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, np.logical_and(test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
                )
            for cut_name, cut in cuts.items():
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, test.loc[:, cut_name].gt(cut).to_numpy() 
                    if 'HH' in cut_name else (
                        test.loc[:, cut_name].lt(cut).to_numpy() 
                        if 'D' in cut_name else 
                        test.loc[:, cut_name].eq(cut).to_numpy()
                    )
                )
            pass_cut_sr_mask = np.logical_and(
                pass_cut_mask,
                np.logical_and(test.loc[:, 'AUX_mass'].gt(115.).to_numpy(), test.loc[:, 'AUX_mass'].lt(135.).to_numpy())
            )
            new_row.append((evtwt_factor * test.loc[pass_cut_sr_mask, 'AUX_eventWeight']).sum())

            if 'H' in sample and 'HH' not in sample: singleH_yield += new_row[-1]
            elif 'H' not in sample: nonRes_yield += new_row[-1]

            not_prev_cut_mask[sample+era_name] = np.logical_and(not_prev_cut_mask[sample+era_name], ~pass_cut_mask)

        new_row.append(singleH_yield); new_row.append(nonRes_yield)

        # for sb_sample_name, sb_sample_arr in zip(['nonRes', 'data'], [sideband_samples, data_samples]):
        #     sb_mask = np.zeros(len(test), dtype=bool)
        #     for sample in sb_sample_arr: sb_mask = np.logical_or(sb_mask, test['AUX_sample_name'].eq(sample))
        #     if sb_sample_name not in not_prev_cut_mask: not_prev_cut_mask[sb_sample_name] = sb_mask

        #     pass_cut_mask = not_prev_cut_mask[sb_sample_name]
        #     if i != 0:
        #         pass_cut_mask = np.logical_and(
        #             pass_cut_mask, np.logical_and(test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
        #         )
        #     for cut_name, cut in cuts.items():
        #         pass_cut_mask = np.logical_and(
        #             pass_cut_mask, test.loc[:, cut_name].gt(cut).to_numpy() 
        #             if 'HH' in cut_name else (
        #                 test.loc[:, cut_name].lt(cut).to_numpy() 
        #                 if 'D' in cut_name else 
        #                 test.loc[:, cut_name].eq(cut).to_numpy()
        #             )
        #         )
        #     pass_cut_sb_mask = np.logical_and(
        #         pass_cut_mask,
        #         np.logical_or(test.loc[:, 'AUX_mass'].lt(115.).to_numpy(), test.loc[:, 'AUX_mass'].gt(135.).to_numpy())
        #     )
        #     if sb_sample_name == 'data':
        #         sb_window_mask = np.logical_and(test.loc[:, 'AUX_mass'].gt(100.).to_numpy(), test.loc[:, 'AUX_mass'].lt(180.).to_numpy())
        #         new_row.append(np.sum(np.logical_and(pass_cut_sb_mask, sb_window_mask)))
        #     if np.sum(pass_cut_sb_mask) != 0:
        #         sb_est_yield = est_yield(test.loc[pass_cut_sb_mask, 'AUX_mass'], test.loc[pass_cut_sb_mask, 'AUX_eventWeight'], [100., 180., 5.], [115., 135.])
        #     else:
        #         sb_est_yield = 0
        #     new_row.append(sb_est_yield)

        for sb_sample_name, sb_sample_arr in zip(['data'], [data_samples]):
            sb_mask = np.logical_and(era_mask, test['AUX_sample_name'].isin(sb_sample_arr))
            if sb_sample_name+era_name not in not_prev_cut_mask: not_prev_cut_mask[sb_sample_name+era_name] = sb_mask

            pass_cut_mask = not_prev_cut_mask[sb_sample_name+era_name]
            if i != 0:
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, np.logical_and(test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
                )
            for cut_name, cut in cuts.items():
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, test.loc[:, cut_name].gt(cut).to_numpy() 
                    if 'HH' in cut_name else (
                        test.loc[:, cut_name].lt(cut).to_numpy() 
                        if 'D' in cut_name else 
                        test.loc[:, cut_name].eq(cut).to_numpy()
                    )
                )
            pass_cut_sb_mask = np.logical_and(
                pass_cut_mask,
                np.logical_or(test.loc[:, 'AUX_mass'].lt(115.).to_numpy(), test.loc[:, 'AUX_mass'].gt(135.).to_numpy())
            )
            sb_window_mask = np.logical_and(test.loc[:, 'AUX_mass'].gt(100.).to_numpy(), test.loc[:, 'AUX_mass'].lt(180.).to_numpy())
            new_row.append(np.sum(np.logical_and(pass_cut_sb_mask, sb_window_mask)))

        # sum_singleH = new_row[1] + sum(new_row[4:11])
        # new_row.append(new_row[2] / (sum_singleH + new_row[11]))
        # new_row.append(new_row[2] / (sum_singleH + new_row[12]))

        table.add_row(new_row)

print(table)


In [ ]:
manos_sample = pd.read_parquet('/uscms/home/tsievert/nobackup/XHYbbgg/v6_merged_scored_events.parquet')
manos_sample['lumievent'] = (manos_sample['lumi'].astype(int).astype(str) + manos_sample['event'].astype(int).astype(str)).astype(int)
manos_sample.keys()

In [ ]:
for sample in pd.unique(manos_sample['sample']):
    sample_mask = manos_sample['sample'].eq(sample)
    # for year in pd.unique(manos_sample['year']):
    #     year_mask = manos_sample['year'].eq(year)
    #     print(sample, ' ', year, ' ', len(manos_sample.loc[np.logical_and(sample_mask, year_mask)]))
    print('num events of ', sample, ' ', 'all years', ' ', len(manos_sample.loc[sample_mask]))
    print('-'*60)

In [ ]:
for sample in pd.unique(test['AUX_sample_name']):
    sample_mask = test['AUX_sample_name'].eq(sample)
    # for year, year_tags in zip(['2016', '2017', '2018', '2022', '2023', '2024', '2025'], [['Run2_2016preVFP/sim', 'Run2_2016postVFP/sim', 'Run2_2016preVFP/data', 'Run2_2016postVFP/data'], ['Run2_2017/sim', 'Run2_2017/data'], ['Run2_2018/sim', 'Run2_2018/data'], ['Run3_2022/sim/preEE', 'Run3_2022/sim/postEE', 'Run3_2022/data'], ['Run3_2023/sim/preBPix', 'Run3_2023/sim/postBPix', 'Run3_2023/data'], ['Run3_2024/sim', 'Run3_2024/data'], ['Run3_2025/sim', 'Run3_2025/data']]):
    #     year_mask = test['AUX_sample_era'].isin(year_tags)
    #     print(sample, ' ', year, ' ', len(test.loc[np.logical_and(sample_mask, year_mask)]))
    print('num events of ', sample, ' ', 'all years', ' ', len(test.loc[sample_mask]))
    print('-'*60)

In [ ]:
import awkward as ak
import numba as nb

@nb.njit
def issubset(subset_candidate_arr, superset_candidate_arr, nonsubset_builder):
    nonsubset_builder.begin_list()
    for sub_elem in subset_candidate_arr:
        found = False
        for super_elem in superset_candidate_arr:
            found = found or (sub_elem == super_elem)
        nonsubset_builder.append(sub_elem * found)
    nonsubset_builder.end_list()
    return nonsubset_builder

In [ ]:
nonsubset_events = issubset(test['lumievent'].to_numpy(), manos_sample['lumievent'].to_numpy(), ak.ArrayBuilder()).snapshot()
# nonsubset_events